# Import

In [1]:
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.oauth2.credentials import Credentials
import io
import os
import os.path
import json
from dotenv import load_dotenv

In [2]:
# Scopes for Google Sheets and Drive
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly', 'https://www.googleapis.com/auth/drive.file']

# NOTE: the notebook may run with cwd in /etl, so load .env from the project root if needed.
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(env_path)

True

In [ ]:
# Authenticate.
# Prefer OAuth token from environment; otherwise fall back to local token.json or service account.
creds = None
oauth_token = os.getenv('OAUTH_TOKEN')

if oauth_token:
    try:
        # Authenticate using the token data
        token_data = json.loads(oauth_token)
        creds = Credentials.from_authorized_user_info(token_data, scopes=SCOPES)
    except Exception as exc:
        raise SystemExit(f'Failed to load OAUTH_TOKEN: {exc}')
else:
    raise SystemExit('No credentials found. Please provide OAUTH_TOKEN, token.json, or service_account.json.')

# Refresh if needed
if not creds.valid:
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        raise SystemExit('Credentials are invalid and cannot be refreshed. Please re-authenticate.')

# Build services
SHEETS_SERVICE = build('sheets', 'v4', credentials=creds)
DRIVE_SERVICE = build('drive', 'v3', credentials=creds)

In [ ]:
def gsheets_to_df(sheet_id, range) -> pd.DataFrame:
    """
    Reads data from a Google Sheets range and returns it as a pandas DataFrame.

    Parameters
    ----------
    sheet_id : str
        The ID of the Google Sheets document.
    range : str
        The A1 notation of the values to retrieve (e.g., 'Sheet1!A1:C100').

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the sheet data, where the first row is used as column headers.
        If no data is found, an empty DataFrame is returned.
    """
    # Initialize the Google Sheets API client
    sheet = SHEETS_SERVICE.spreadsheets()

    # Fetch values from the specified spreadsheet and range
    result = sheet.values().get(
        spreadsheetId=sheet_id,
        range=range
    ).execute()

    # Extract the values from the response (list of rows)
    values = result.get('values', [])

    # Convert to DataFrame:
    # - First row becomes column names
    # - Remaining rows become data
    return pd.DataFrame(data=values[1:], columns=values[0])

In [10]:
def df_to_drive_csv(df, filename, folder_id=None):
    """
    Saves DataFrame to CSV in Google Drive.
    """
    csv_string = df.to_csv(index=False)
    file_metadata = {'name': filename}
    if folder_id:
        file_metadata['parents'] = [folder_id]
    media = MediaIoBaseUpload(io.BytesIO(csv_string.encode('utf-8')), mimetype='text/csv')
    file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    return file.get('id')

In [16]:
# The ID and range of a sample spreadsheet.
SAMPLE_SPREADSHEET_ID = '1pZAAw9L8FvPQnad5ejfM7gu_JCeuwuEy7AqGiA5wKy0'
SAMPLE_RANGE_NAME = 'palpites_fg!A1:ET20'

df = gsheets_to_df(SAMPLE_SPREADSHEET_ID, SAMPLE_RANGE_NAME, sheets_service)

In [18]:
df

,Carimbo de data/hora,Nome,Deixe uma foto sua aqui,Campeão,Vice,Artilheiro,México x África do Sul [México],México x África do Sul [África do Sul],Coreia do Sul x Tchéquia [Coreia do Sul],Coreia do Sul x Tchéquia [Tchéquia],...,Gana x Panamá [Gana],Gana x Panamá [Panamá],Inglaterra x Gana [Inglaterra],Inglaterra x Gana [Gana],Panamá x Croácia [Panamá],Panamá x Croácia [Croácia],Panamá x Inglaterra [Panamá],Panamá x Inglaterra [Inglaterra],Croácia x Gana [Croácia],Croácia x Gana [Gana]
0,06/03/2026 23:36:41,ferolifil,https://drive.google.com/open?id=1q8JjMZeWEmxL...,Brasil,Espanha,Endrick,0,0,1,1,...,1,0,0,1,3,2,1,3,1,3
1,20/02/2026 00:00:00,washington,https://drive.google.com/file/d/1SS77bKTZFoWj3...,Brasil,Alemanha,Messi,3,3,1,2,...,3,3,0,0,1,0,2,1,1,4
2,10/02/2026 00:00:00,francisca,https://drive.google.com/file/d/1NMiEm61hLaWl0...,Brasil,Croácia,Neymar,2,2,0,4,...,0,0,3,0,3,2,2,2,1,3
3,23/02/2026 00:00:00,ana nath,https://drive.google.com/file/d/1tSg_c0_3qqaPl...,Brasil,Austrália,Van Djik,4,0,0,0,...,3,2,2,2,3,2,3,2,3,1
